In [6]:
import numpy as np
import pandas as pd
import yt
import trident
import matplotlib.pyplot as plt

from yt.visualization.volume_rendering.off_axis_projection import off_axis_projection

CUTOUT_H5 = "/scratch/tsingh65/TNG50-1_snap99/out_sub_488530/cutout_ALLFIELDS_sphere_2p1Rvir_sub488530.hdf5"
RAYS_CSV  = "/scratch/tsingh65/m61-tng/outputs/sid488530/rays_and_recipes_sid488530_snap99_L4Rvir/rays_sid488530.csv"

ALPHAS = list(range(0, 11))   # 0..10
ALPHA_TOL_DEG = 0.5

WIDTH_KPC = 100.0
DEPTH_KPC = 100.0

RES = 512
RES2 = (RES, RES)             # <-- FIX

HI_ION = "H I"
HI_FIELD = ("gas", "H_p0_number_density")

def _unit_vec(v):
    v = np.asarray(v, float)
    n = np.linalg.norm(v)
    if not np.isfinite(n) or n == 0:
        raise ValueError("bad vector")
    return v / n

def _north_vector(normal):
    z = np.array([0.0, 0.0, 1.0])
    if abs(np.dot(_unit_vec(normal), z)) > 0.95:
        z = np.array([0.0, 1.0, 0.0])
    north = np.cross(normal, z)
    if np.linalg.norm(north) == 0:
        north = np.array([1.0, 0.0, 0.0])
    return _unit_vec(north)

from yt.visualization.volume_rendering.off_axis_projection import off_axis_projection
import numpy as np

def _offaxis_project(ds, center, normal, north, width_kpc, depth_kpc, res2, field):
    """
    Always pass width as a 3-vector to satisfy yt's expectations.
    width_kpc sets the image plane (x,y) size; depth_kpc sets LOS integration thickness.
    """
    # 3-vector width: (xwidth, ywidth, zwidth)
    width_vec = ds.arr([width_kpc, width_kpc, depth_kpc], "kpc")

    # Use the positional signature (most stable across yt versions)
    img = off_axis_projection(
        ds,                 # data_source
        center,             # center
        normal,             # normal_vector
        width_vec,          # width (3-vector)
        res2,               # resolution (nx, ny)
        field,              # item (field)
        north_vector=north  # orientation
    )
    return img

ds = yt.load(CUTOUT_H5)
trident.add_ion_fields(ds, ions=[HI_ION])

df = pd.read_csv(RAYS_CSV)
df = df[np.isfinite(df["alpha_deg"].values)]

for a in ALPHAS:
    sel = df[np.abs(df["alpha_deg"].values - float(a)) <= ALPHA_TOL_DEG].copy()
    if sel.empty:
        print(f"[SKIP] alpha~{a}: no rays within +/-{ALPHA_TOL_DEG} deg")
        continue

    print(f"[ALPHA {a}] {len(sel)} ray(s)")

    for _, row in sel.iterrows():
        sightline_id = str(row.get("sightline_id", "SL"))
        mode         = str(row.get("mode", "mode"))
        alpha_tag    = int(round(float(row["alpha_deg"])))

        p0 = np.array([row["p0_X_ckpch_abs"], row["p0_Y_ckpch_abs"], row["p0_Z_ckpch_abs"]], float)
        p1 = np.array([row["p1_X_ckpch_abs"], row["p1_Y_ckpch_abs"], row["p1_Z_ckpch_abs"]], float)

        normal = _unit_vec(p1 - p0)
        north  = _north_vector(normal)
        center = ds.arr(0.5 * (p0 + p1), "code_length")

        img = _offaxis_project(
            ds=ds,
            center=center,
            normal=normal,
            north=north,
            width_kpc=WIDTH_KPC,
            depth_kpc=DEPTH_KPC,
            res2=RES2,
            field=HI_FIELD,
        )

        arr = np.asarray(img)
        plt.figure(figsize=(6, 5))
        plt.imshow(arr, origin="lower")
        plt.title(f"H I off-axis proj | alpha={alpha_tag} | {mode} | {sightline_id}")
        plt.colorbar(label=str(img.units))
        plt.tight_layout()
        plt.show()

yt : [INFO     ] 2026-02-13 17:50:56,035 Calculating time from 1.000e+00 to be 4.356e+17 seconds
yt : [INFO     ] 2026-02-13 17:50:56,074 Parameters: current_time              = 4.355810528213311e+17 s
yt : [INFO     ] 2026-02-13 17:50:56,075 Parameters: domain_dimensions         = [1 1 1]
yt : [INFO     ] 2026-02-13 17:50:56,076 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-02-13 17:50:56,077 Parameters: domain_right_edge         = [35000. 35000. 35000.]
yt : [INFO     ] 2026-02-13 17:50:56,078 Parameters: cosmological_simulation   = True
yt : [INFO     ] 2026-02-13 17:50:56,079 Parameters: current_redshift          = 2.220446049250313e-16
yt : [INFO     ] 2026-02-13 17:50:56,083 Parameters: omega_lambda              = 0.6911
yt : [INFO     ] 2026-02-13 17:50:56,083 Parameters: omega_matter              = 0.3089
yt : [INFO     ] 2026-02-13 17:50:56,090 Parameters: omega_radiation           = 0.0
yt : [INFO     ] 2026-02-13 17:50:56,091 Parameters: hubble_con

[ALPHA 0] 2 ray(s)
[ALPHA 1] 2 ray(s)
[ALPHA 2] 2 ray(s)
[ALPHA 3] 2 ray(s)
[ALPHA 4] 2 ray(s)
[ALPHA 5] 2 ray(s)
[ALPHA 6] 2 ray(s)
[ALPHA 7] 2 ray(s)
[ALPHA 8] 2 ray(s)
[ALPHA 9] 2 ray(s)
[ALPHA 10] 2 ray(s)


/tmp/ipykernel_3620637/2685403562.py:101: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(6, 5))


In [7]:
import os
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt

# -------------------------
# EDIT THESE PATHS
# -------------------------
CUTOUT_H5 = "/scratch/tsingh65/TNG50-1_snap99/out_sub_488530/cutout_ALLFIELDS_sphere_2p1Rvir_sub488530.hdf5"
RAYS_CSV  = "/scratch/tsingh65/m61-tng/outputs/sid488530/rays_and_recipes_sid488530_snap99_L4Rvir/rays_sid488530.csv"
OUTDIR    = "/scratch/tsingh65/m61-tng/outputs/sid488530/rays_and_spectra_sid488530_snap99_L4Rvir/HI_manual_projections_alpha0_10"

# map size
WIDTH_KPC = 100.0     # requested: 100 kpc wide
DEPTH_KPC = 100.0     # choose a slab depth; change if you want
RES       = 512       # pixels per side

ALPHAS = list(range(0, 11))  # 0..10 inclusive
ALPHA_TOL_DEG = 0.5          # match rays within +/- tolerance

# -------------------------
# HDF5 helpers (no yt)
# -------------------------
def _h5_list_paths(f: h5py.File):
    out = []
    def _walk(g, prefix=""):
        for k in g.keys():
            obj = g[k]
            p = f"{prefix}/{k}"
            out.append(p)
            if isinstance(obj, h5py.Group):
                _walk(obj, p)
    _walk(f, "")
    return out

def _get_header_attr(f: h5py.File, key: str):
    # TNG cutouts often store attrs in /Header
    if "Header" in f and key in f["Header"].attrs:
        return f["Header"].attrs[key]
    # fallback: some files store at root
    if key in f.attrs:
        return f.attrs[key]
    return None

def _get_length_conv_to_kpc(f: h5py.File):
    """
    Best-effort conversion for TNG-style cutouts:
      Coordinates typically in comoving kpc/h (ckpc/h).
      physical kpc = ckpc/h * a / h   (if comoving)
    We do NOT assume this is true; we only apply if the needed header attrs exist.
    If attrs are missing, we return scale=1 (treat as already-kpc).
    """
    unit_len_cm = _get_header_attr(f, "UnitLength_in_cm")
    hubble      = _get_header_attr(f, "HubbleParam")
    time_a      = _get_header_attr(f, "Time")  # scale factor a

    kpc_cm = 3.0856775814913673e21

    # If UnitLength_in_cm exists, convert code length to kpc.
    # If also cosmological attrs exist, we also apply a/h for physical.
    if unit_len_cm is None:
        return 1.0, "kpc (assumed)"

    code_to_kpc = float(unit_len_cm) / kpc_cm

    # Only apply a/h if both exist; otherwise do nothing extra.
    if (hubble is not None) and (time_a is not None):
        code_to_kpc *= float(time_a) / float(hubble)
        return code_to_kpc, "kpc (using UnitLength_in_cm * a/h)"
    return code_to_kpc, "kpc (using UnitLength_in_cm only)"

def _unit_vec(v):
    v = np.asarray(v, float)
    n = np.linalg.norm(v)
    if not np.isfinite(n) or n == 0:
        raise ValueError("bad vector")
    return v / n

def _north_vector(normal):
    z = np.array([0.0, 0.0, 1.0])
    n = _unit_vec(normal)
    if abs(np.dot(n, z)) > 0.95:
        z = np.array([0.0, 1.0, 0.0])
    north = np.cross(n, z)
    if np.linalg.norm(north) == 0:
        north = np.array([1.0, 0.0, 0.0])
    return _unit_vec(north)

def _pick_one_ray_for_alpha(df: pd.DataFrame, alpha_deg: float, tol: float):
    d = df[np.isfinite(df["alpha_deg"].values)]
    d = d[np.abs(d["alpha_deg"].values - alpha_deg) <= tol]
    if d.empty:
        return None
    # deterministic: choose first row in file order
    return d.iloc[0]

def _load_gas_arrays(f: h5py.File):
    """
    Minimal: loads gas coordinates and some candidate fields.
    We do not assume names beyond common TNG cutout conventions:
      /PartType0/Coordinates (required)
      /PartType0/Volume OR (/PartType0/Masses AND /PartType0/Density) (optional)
      One of:
        - /PartType0/H_p0_number_density   (rare; usually NOT in cutout)
        - /PartType0/HI_NumberDensity      (if you wrote it yourself)
        - /PartType0/NeutralHydrogenAbundance + /PartType0/Density (requires more info; not assumed)
    """
    g = f.get("PartType0", None)
    if g is None:
        raise RuntimeError("Missing group /PartType0 in cutout. Cannot proceed without gas group.")

    if "Coordinates" not in g:
        raise RuntimeError("Missing /PartType0/Coordinates in cutout. Cannot proceed.")

    coords = g["Coordinates"][...]  # (N,3)

    # volume (best) or mass+density (fallback) for an effective dl estimate
    vol = None
    if "Volume" in g:
        vol = g["Volume"][...]
    elif ("Masses" in g) and ("Density" in g):
        m = g["Masses"][...]
        rho = g["Density"][...]
        # guard against division-by-zero
        vol = np.where(rho > 0, m / rho, np.nan)

    # Try to find HI number density field directly (no assumptions).
    nHI = None
    nHI_path_used = None
    for cand in ["H_p0_number_density", "HI_NumberDensity", "HI_number_density", "NeutralHydrogenNumberDensity"]:
        if cand in g:
            nHI = g[cand][...]
            nHI_path_used = f"/PartType0/{cand}"
            break

    if nHI is None:
        # Do not try to derive HI from density without explicit input fields/constants.
        # Fail loudly with instructions on what exists.
        paths = _h5_list_paths(f)
        hint = [p for p in paths if ("HI" in p or "H_p0" in p or "NeutralHydrogen" in p)]
        raise RuntimeError(
            "No direct HI number-density dataset found in the cutout under /PartType0.\n"
            "Expected one of: H_p0_number_density, HI_NumberDensity, HI_number_density, NeutralHydrogenNumberDensity.\n"
            f"Found these HI-like paths:\n{hint}"
        )

    return coords, vol, nHI, nHI_path_used

def _project_hi_map(coords_kpc, vol_kpc3, nHI, center_kpc, normal, north, width_kpc, depth_kpc, res):
    """
    Very basic projection without yt:
      - rotate coords into (east, north, normal) basis
      - select slab within width x width x depth around center
      - estimate per-element dl:
          if vol provided: dl ~ vol^(1/3)
          else: dl = depth/res  (uniform)
      - bin onto 2D grid with weights = nHI * dl

    Output: 2D array (res,res) in arbitrary units consistent with your nHI and length.
    """
    normal = _unit_vec(normal)
    north  = _unit_vec(north)
    east   = _unit_vec(np.cross(north, normal))

    rel = coords_kpc - center_kpc[None, :]
    x = rel @ east
    y = rel @ north
    z = rel @ normal

    half_w = 0.5 * width_kpc
    half_d = 0.5 * depth_kpc

    m = (np.abs(x) <= half_w) & (np.abs(y) <= half_w) & (np.abs(z) <= half_d)
    if not np.any(m):
        return np.zeros((res, res), dtype=float)

    x = x[m]; y = y[m]
    n = np.asarray(nHI)[m].astype(float)

    if vol_kpc3 is not None:
        v = np.asarray(vol_kpc3)[m].astype(float)
        dl = np.cbrt(v)  # kpc
        dl = np.where(np.isfinite(dl) & (dl > 0), dl, 0.0)
    else:
        dl = np.full_like(n, depth_kpc / float(res), dtype=float)

    w = n * dl  # "column-like" weight (still basic)

    # bin edges
    edges = np.linspace(-half_w, half_w, res + 1)
    H, _, _ = np.histogram2d(y, x, bins=(edges, edges), weights=w)  # y first => rows

    return H

# -------------------------
# Main
# -------------------------
os.makedirs(OUTDIR, exist_ok=True)

df = pd.read_csv(RAYS_CSV)
req_cols = ["alpha_deg", "p0_X_ckpch_abs", "p0_Y_ckpch_abs", "p0_Z_ckpch_abs",
            "p1_X_ckpch_abs", "p1_Y_ckpch_abs", "p1_Z_ckpch_abs"]
for c in req_cols:
    if c not in df.columns:
        raise RuntimeError(f"Missing required column '{c}' in rays CSV: {RAYS_CSV}")

with h5py.File(CUTOUT_H5, "r") as f:
    L_to_kpc, L_label = _get_length_conv_to_kpc(f)
    coords, vol, nHI, nHI_path = _load_gas_arrays(f)

    # Convert coords to kpc using header-driven scale if available.
    coords_kpc = coords.astype(float) * float(L_to_kpc)

    # Convert volume to kpc^3 if we can infer it from the same length conversion.
    vol_kpc3 = None
    if vol is not None:
        vol_kpc3 = vol.astype(float) * (float(L_to_kpc) ** 3)

    print(f"[INFO] length conversion: {L_to_kpc} -> {L_label}")
    print(f"[INFO] using HI number density from: {nHI_path}")
    print(f"[INFO] gas elements: {coords_kpc.shape[0]}")

    for a in ALPHAS:
        row = _pick_one_ray_for_alpha(df, float(a), ALPHA_TOL_DEG)
        if row is None:
            print(f"[SKIP] alpha={a}: no ray within tol={ALPHA_TOL_DEG} deg")
            continue

        p0 = np.array([row["p0_X_ckpch_abs"], row["p0_Y_ckpch_abs"], row["p0_Z_ckpch_abs"]], float) * float(L_to_kpc)
        p1 = np.array([row["p1_X_ckpch_abs"], row["p1_Y_ckpch_abs"], row["p1_Z_ckpch_abs"]], float) * float(L_to_kpc)

        normal = _unit_vec(p1 - p0)
        north  = _north_vector(normal)
        center = 0.5 * (p0 + p1)

        img = _project_hi_map(
            coords_kpc=coords_kpc,
            vol_kpc3=vol_kpc3,
            nHI=nHI,
            center_kpc=center,
            normal=normal,
            north=north,
            width_kpc=WIDTH_KPC,
            depth_kpc=DEPTH_KPC,
            res=RES,
        )

        # show in-notebook + save
        plt.figure(figsize=(6, 5))
        plt.imshow(img, origin="lower", extent=[-WIDTH_KPC/2, WIDTH_KPC/2, -WIDTH_KPC/2, WIDTH_KPC/2])
        plt.colorbar(label="Σ n(HI)·dl (basic units)")
        plt.title(f"H I (manual binning) | alpha={a} | width={WIDTH_KPC} kpc | depth={DEPTH_KPC} kpc")
        plt.xlabel("x (kpc)")
        plt.ylabel("y (kpc)")
        out = os.path.join(OUTDIR, f"HI_alpha{a:02d}_W{int(WIDTH_KPC)}kpc_D{int(DEPTH_KPC)}kpc.png")
        plt.savefig(out, dpi=200, bbox_inches="tight")
        plt.show()
        plt.close()

print(f"[OK] wrote images to: {OUTDIR}")

RuntimeError: No direct HI number-density dataset found in the cutout under /PartType0.
Expected one of: H_p0_number_density, HI_NumberDensity, HI_number_density, NeutralHydrogenNumberDensity.
Found these HI-like paths:
['/PartType0/NeutralHydrogenAbundance']

## Checking Velocity Maps